# Barlow Twins for NIR Spectroscopy — PyTorch Example

Semi-supervised regression predicting dry matter content (DM) from NIR spectra,
using a small labeled subset plus a large unlabeled pool.

This notebook is the PyTorch counterpart of `kiwifruit_example.ipynb`.

## 1. Install & Import

In [ ]:
# Install the package (run once)
# !pip install -e ..[torch]   # from repo root

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader

from barlow_twins_nir_torch import (
    load_kiwifruit,
    remove_outliers,
    normalize_features,
    make_paired_views,
    PairedNIRDataset,
    LabeledNIRDataset,
    make_labeled_dataset,
    make_validation_dataloaders,
    BarlowRegressionModel,
    SupervisedModel,
    train_barlow,
    train_supervised,
    compute_metrics,
    plot_predictions,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## 2. Load & Preprocess

In [ ]:
DATA_PATH = '/data/kiwifruit_dat.csv'  # adjust path as needed
X_LOWER = 'X402'
X_UPPER = 'X1065'
TARGET_COL = 'DM'

kiwi = load_kiwifruit(DATA_PATH, dm_cutoff=7, min_readings=2)
print(f'Loaded: {len(kiwi)} rows, {kiwi["sample_id"].nunique()} unique samples')

In [ ]:
kiwi = remove_outliers(kiwi, x_lower=X_LOWER, x_upper=X_UPPER,
                       n_components=20, threshold=1200)
print(f'After outlier removal: {len(kiwi)} rows')

In [ ]:
repeated_kiwi_x, repeated_kiwi_y = make_paired_views(kiwi, X_LOWER, X_UPPER)
print(f'Paired views: {len(repeated_kiwi_x)} rows each')

In [ ]:
train_mask_x = repeated_kiwi_x['Dataset'] == 'Training'
train_mask_y = repeated_kiwi_y['Dataset'] == 'Training'

features_x = repeated_kiwi_x.loc[:, X_LOWER:X_UPPER]
features_y = repeated_kiwi_y.loc[:, X_LOWER:X_UPPER]

features_x_norm = normalize_features(features_x, train_mask_x)
features_y_norm = normalize_features(features_y, train_mask_y)

N_FEATURES = features_x_norm.shape[1]
print(f'Feature dimensions: {N_FEATURES}')

## 3. Build DataLoaders

In [ ]:
NSAMP = 100       # number of labeled training samples
BATCH_SIZE = 4000 # unlabeled batch size
ENC_SIZES = (16,)
REG_SIZES = (1,)

In [ ]:
# Unlabeled DataLoader — all training paired views
unlabeled_ds = PairedNIRDataset(features_x_norm, features_y_norm, mask=train_mask_x)
unlabeled_loader = DataLoader(unlabeled_ds, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
# Labeled DataLoader — small subset
labeled_loader, tail_ids, features_sub_norm, target_sub = make_labeled_dataset(
    kiwi, nsamp=NSAMP, x_lower=X_LOWER, x_upper=X_UPPER,
    target_col=TARGET_COL, batch_size=BATCH_SIZE,
)
print(f'Labeled samples: {len(features_sub_norm)}')

In [ ]:
# Validation and test DataLoaders
val_loader, test_loader = make_validation_dataloaders(
    features_x_norm, features_y_norm,
    repeated_kiwi_x, repeated_kiwi_y,
    target_col=TARGET_COL, batch_size=4000,
)

## 4. Train Barlow Twins Model

In [ ]:
barlow_model = BarlowRegressionModel(
    input_size=N_FEATURES,
    enc_sizes=ENC_SIZES,
    reg_sizes=REG_SIZES,
    loss_weight=(10.5, 0.5, 0.5, 0.),
    barlow_lambda=1 / 15,
    activation='linear',
    conv1=True,
    preprocess=3,
)

history_barlow = train_barlow(
    barlow_model,
    unlabeled_loader=unlabeled_loader,
    labeled_loader=labeled_loader,
    val_loader=val_loader,
    epochs=1000,
    lr=0.005,
    clip_value=1.0,
    patience=50,
    checkpoint_path='model_barlow.pt',
    device=DEVICE,
)

## 5. Train Supervised Baseline

In [ ]:
# Normalize single-view features for the supervised baseline
features_all = kiwi.loc[:, X_LOWER:X_UPPER]
train_mask_all = kiwi['Dataset'] == 'Training'
features_norm = normalize_features(features_all, train_mask_all)

# Validation data as tensors
val_mask_all = kiwi['Dataset'] == 'Validation'
val_data = (
    features_norm.loc[val_mask_all].values,
    kiwi.loc[val_mask_all, TARGET_COL].values,
)

In [ ]:
from barlow_twins_nir_torch import LabeledNIRDataset

sup_train_ds = LabeledNIRDataset(features_sub_norm, target_sub)
sup_train_loader = DataLoader(sup_train_ds, batch_size=500, shuffle=True)

sup_model = SupervisedModel(
    input_size=N_FEATURES,
    enc_sizes=ENC_SIZES,
    reg_sizes=REG_SIZES,
    activation='linear',
    conv1=True,
    preprocess=3,
)

history_sup = train_supervised(
    sup_model,
    train_loader=sup_train_loader,
    val_data=val_data,
    epochs=1000,
    lr=0.005,
    patience=150,
    checkpoint_path='model_supervised.pt',
    device=DEVICE,
)

## 6. Evaluate & Compare

In [ ]:
barlow_model.eval()
non_train_mask = repeated_kiwi_x['Dataset'] != 'Training'
x_eval = torch.tensor(
    features_x_norm.loc[non_train_mask].values, dtype=torch.float32
).to(DEVICE)

with torch.no_grad():
    y_pred_barlow = barlow_model(x_eval).squeeze(-1).cpu().numpy()
y_true_barlow = repeated_kiwi_x.loc[non_train_mask, TARGET_COL].values

metrics_barlow = compute_metrics(y_true_barlow, y_pred_barlow)
print('Barlow Twins — RMSE: {rmse:.3f}  R²: {r2:.3f}'.format(**metrics_barlow))

In [ ]:
sup_model.eval()
non_train_mask_kiwi = kiwi['Dataset'] != 'Training'
x_eval_sup = torch.tensor(
    features_norm.loc[non_train_mask_kiwi].values, dtype=torch.float32
).to(DEVICE)

with torch.no_grad():
    y_pred_sup = sup_model(x_eval_sup).squeeze(-1).cpu().numpy()
y_true_sup = kiwi.loc[non_train_mask_kiwi, TARGET_COL].values

metrics_sup = compute_metrics(y_true_sup, y_pred_sup)
print('Supervised    — RMSE: {rmse:.3f}  R²: {r2:.3f}'.format(**metrics_sup))

In [ ]:
plot_predictions(y_true_barlow, y_pred_barlow,
                 title=f'Barlow Twins — PyTorch (n={NSAMP})',
                 save_path='barlow_predictions_torch.png')

plot_predictions(y_true_sup, y_pred_sup,
                 title=f'Supervised Baseline — PyTorch (n={NSAMP})',
                 save_path='supervised_predictions_torch.png')